# DATASCI 530 – Project 1: MPIDR Research Projects

**Source:** Max Planck Institute for Demographic Research (MPIDR), research unit project lists, https://www.demogr.mpg.de/en/research_6120/  
**Collected on:** _fill in_  
**Observations:** _fill in_ projects across _fill in_ research units; _fill in_ project-member rows  
**Restrictions:** English-language site only; current projects listed on each unit's Projects tab; 1.5-second delay between requests; metadata only (description length, not full text). robots.txt and Imprint checked on _fill in date_.

## Setup

In [ ]:
import re, time
from datetime import date
import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://www.demogr.mpg.de"
DELAY = 1.5
HEADERS = {"User-Agent": "DATASCI530 student project (your.email@emory.edu)"}

def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    time.sleep(DELAY)
    return BeautifulSoup(r.text, "html.parser")

def absolute(href):
    return href if href.startswith("http") else BASE + href

## Step 1: Find all research units
Units are the labs and research groups linked from the Research menu (two levels below `/research_6120/`).

In [ ]:
UNIT_PATTERN = re.compile(r"/en/research_6120/([^/]+)/([^/]+)/?$")

soup = get_soup(BASE + "/en/research_6120/")
units = {}
for a in soup.find_all("a", href=True):
    m = UNIT_PATTERN.search(a["href"])
    if m:
        url = absolute(a["href"]).rstrip("/") + "/"
        units[url] = {"unit_url": url, "unit_group": m.group(1), "unit": a.get_text(strip=True)}
units = pd.DataFrame(units.values())
units

## Step 2: Loop over each unit's project list
For each unit, open its Projects tab and record every project link and the research area heading it sits under. Units without a Projects tab are recorded and skipped.

In [ ]:
project_links, skipped = [], []
for _, u in units.iterrows():
    try:
        s = get_soup(u["unit_url"] + "projects/")
    except requests.HTTPError:
        skipped.append(u["unit"])
        continue
    for a in s.find_all("a", href=True, string=re.compile("Project details")):
        area = a.find_previous("h2")
        project_links.append({"project_url": absolute(a["href"]),
                              "unit": u["unit"], "unit_group": u["unit_group"],
                              "research_area": area.get_text(strip=True) if area else None})

projects = pd.DataFrame(project_links)
print(len(projects), "project links;", "skipped units:", skipped)

## Step 3: Navigate to each project page
Extracts title, MPIDR team members, external collaborators, description length, keywords, regions, and linked publications.

In [ ]:
PUB_PATTERN = re.compile(r"/publications_1904/[^/]+/[^/]+_\d+/?$")

def content_lines(start):
    """Text lines from the project title down to 'Back to Project List'."""
    out = []
    for s in start.find_all_next(string=True):
        t = s.strip()
        if t == "Back to Project List":
            break
        if t:
            out.append(t)
    return out

def after_label(lines, label):
    """Line(s) following a label such as 'Region keywords:'."""
    if label not in lines:
        return None
    nxt = lines[lines.index(label) + 1]
    return None if nxt.endswith(":") else nxt

def parse_project(url):
    s = get_soup(url)
    h2 = s.find("h2")                              # project title
    team = h2.find_next("p")                       # VERIFY with Inspect
    members = team.select('a[href*="staff_directory"]') if team else []
    team_text = team.get_text(" ", strip=True) if team else ""
    collab = re.split(r"[Cc]ollaboration with", team_text)
    collab = collab[1] if len(collab) > 1 else ""
    institutions = re.findall(r"\(([^)]*)\)", collab)
    ext_names = [n for n in re.sub(r"\([^)]*\)", "", collab).split(",") if n.strip()]

    desc = team.find_next("p") if team else None
    lines = content_lines(h2)
    pub_lines = lines[lines.index("Publications"):] if "Publications" in lines else []
    years = [int(y) for y in re.findall(r"\((\d{4})\)", " ".join(pub_lines))]
    pubs = [a for a in h2.find_all_next("a", href=True) if PUB_PATTERN.search(a["href"])]

    return {"project_url": url,
            "title": h2.get_text(strip=True),
            "members": "; ".join(a.get_text(strip=True) for a in members),
            "member_urls": "; ".join(absolute(a["href"]) for a in members),
            "n_members": len(members),
            "n_external": len(ext_names),
            "n_external_institutions": len(institutions),
            "description_words": len(desc.get_text().split()) if desc else None,
            "keywords": after_label(lines, "Research Keywords:"),
            "regions": after_label(lines, "Region keywords:"),
            "n_publications": len(pubs),
            "n_open_access": len(h2.find_all_next("img", title="Open Access")),
            "latest_pub_year": max(years) if years else None}

details = pd.DataFrame([parse_project(u) for u in projects["project_url"].unique()])

## Step 4: Build and save the datasets

In [ ]:
out = projects.merge(details, on="project_url", how="left")
out["collected_on"] = date.today().isoformat()
out.to_csv("mpidr_projects.csv", index=False)

# One row per project x MPIDR member (useful for finding researchers by topic)
members = (out[out["n_members"] > 0]
           .assign(member=out["members"].str.split("; "), member_url=out["member_urls"].str.split("; "))
           .explode(["member", "member_url"])
           [["title", "unit", "research_area", "member", "member_url"]])
members.to_csv("mpidr_project_members.csv", index=False)

print(len(out), "project rows;", out["project_url"].nunique(), "unique projects;", len(members), "member rows")

## Results
### Observations and missing values

In [ ]:
summary = pd.DataFrame({"Non-missing": out.notna().sum(), "Missing": out.isna().sum()})
summary

### Mean and variance of numeric variables

In [ ]:
num = ["n_members", "n_external", "n_external_institutions", "description_words",
       "n_publications", "n_open_access", "latest_pub_year"]
out[num].agg(["mean", "var"]).T.round(2)

### Quality checks
_Document here, for example:_
- Projects listed under more than one unit (`project_url` duplicates), kept once per unit or deduplicated
- Units with no Projects tab (`skipped`)
- Projects with no MPIDR members or no publications parsed: spot-check a few by hand
- External collaborator counts are approximate (parsed from free text); compare a sample against the pages

## Discussion
_Brief summary of what was collected and its limitations._